Prepare covariates:

- SEDAC score
- Subnational gender gaps data (geomatch Insta areas to subnational areas)
- Descriptive table by location (Response_rate, total sample size, avg sample size per subgroup)
- marginal effects as a table
- Wellbeing scores before/after for control/group/individual (disaggregated by gender)
- Robustness checks: Add post-stratification weights, aggregated analysis by location

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
import json
import time
import pycountry
from pathlib import Path
from rasterstats import zonal_stats

### Load data

In [ ]:
df_raw = pd.read_csv('../input/Wellbeing_Results_all.csv')
df_rel_raw = pd.read_csv('../input/areas_dominant_religion_with_reach.csv', sep = ';')
df_metadata = gpd.read_parquet('../input/ad_metadata.parquet')

### Assign buffers to Instagram clusters

In [ ]:
df_time = df_raw[['Location', 'Day', 'Time']].drop_duplicates()

In [ ]:
df_rel = (df_rel_raw
          .loc[df_rel_raw.place_name.isin(df_raw.Location.unique()), ['place_name', 'dominant']]
          .rename(columns = {'place_name':'Location', 'dominant':'Religion'})
          .reset_index(drop = True))

In [ ]:
df_raw = df_raw.merge(df_rel, on = 'Location', how = 'left')

In [ ]:
df_metadata['ID'] = df_metadata['adset_id'].astype(int)
df_metadata = df_metadata.drop(columns = ['age', 'gender'])

In [ ]:
df_raw = df_raw.merge(df_metadata, on = 'ID', how = 'left')

first_run = ['Galeshewe', 'Windhoek' ,'Tanana Ambany', 'Aba', 'Yaoundé', 'Grombalia', 'Kati', 'Kipé', 'Al-Qanatir al-Chairiyya', 'Birkhadem']
second_run = ['Santa Catarina', 'Njikoka', 'Bojanala Platinum', 'Kiambu County', 'Aïn Sebaâ']

df_raw['Radius'] = 20
df_raw.loc[df_raw['Location'].isin(first_run), 'Radius'] = 10

df_raw = df_raw[['Location', 'latitude', 'longitude', 'Radius']].drop_duplicates().copy()

Remove surveys without spatiotemporal information (affects just one survey without responses)

In [ ]:
df_raw.dropna(subset = ['start_time', 'end_time', 'geometry'], inplace = True)

In [ ]:
gdf_raw = gpd.GeoDataFrame(
    df_raw,
    geometry=df_raw.geometry,
    crs="EPSG:4326")

In [ ]:
gdf_raw['geometry'] = (gdf_raw
                       .geometry
                       .to_crs(epsg=3857)
                       .buffer(gdf_raw['radius'] * 1000)
                       .to_crs(epsg=4326))

### Intersect with GADM shapes

Asign catchment area of survey to subnational GADM area by largest area intersect. Both the shapes and the covariates are derived from the Subnational Gender Gaps paper from Casey Breen et al.: https://www.pnas.org/doi/10.1073/pnas.2416624122#data-availability

In [ ]:
world_gdf = gpd.read_file("/path/to/external_data/Subnational_Gaps/gadm_410-levels.gpkg", layer = 'ADM_1').to_crs(gdf_raw.crs)

In [ ]:
intersections = gpd.overlay(
    gdf_raw[['Location', 'geometry']],
    world_gdf[['GID_1', 'GID_0', 'geometry']],
    how='intersection'
)
intersections['area'] = intersections.area

In [ ]:
largest_country = (
    intersections.sort_values('area', ascending=False)
    .drop_duplicates(subset='Location')
)

In [ ]:
gdf = (gdf_raw
       .merge(largest_country[['Location', 'GID_0', 'GID_1']], on='Location', how='left')
       .rename(columns = {'GID_0':'gid_0', 'GID_1':'gid_1'}))

## Read covariate data

In [ ]:
cov_raw = pd.read_csv('/path/to/external_data/Subnational_Gaps/master_data_file_march12_with_national.csv')

In [ ]:
cov_df = (cov_raw
          .query("(year >= 2023)")
          .groupby(['gid_0', 'gid_1'])[cov_raw.select_dtypes(include="number").columns]
          .mean()
          .reset_index())

In [ ]:
gdf_cov = gdf.merge(cov_df, on = ['gid_0', 'gid_1'], how = 'left')

## Add Weather

## Weather

Using the open meteo historical weather API: https://open-meteo.com/en/docs/historical-weather-api

In [ ]:
df_time= (gdf_cov[['Location', 'start_time', 'end_time', 'longitude', 'latitude']]
          .drop_duplicates()
          .dropna()
          .reset_index(drop = True)
          .copy())

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"

hourly_vars = ",".join([
    "temperature_2m",
    "precipitation",
    "rain",
    "cloud_cover_high",
    "cloud_cover_mid",
    "cloud_cover_low",
    "cloud_cover",
    "surface_pressure",
    "wind_speed_10m",
    "wind_gusts_10m",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm",
    "relative_humidity_2m",
#    "weather_code",
    "sunshine_duration"
])

In [ ]:
rows = []

df_weather = pd.DataFrame()

for row in tqdm(df_time.itertuples(index=False)):
    params = {
        "latitude": row.latitude,
        "longitude": row.longitude,
        "start_date": row.start_time.strftime("%Y-%m-%d"),
        "end_date": row.end_time.strftime("%Y-%m-%d"),
        "hourly": hourly_vars,
        "timezone": "auto",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    temp = pd.DataFrame(data["hourly"])
    temp["time"] = pd.to_datetime(temp["time"]).dt.tz_localize(row.start_time.tz)

    df_row = temp.loc[
        (temp["time"] >= row.start_time) &
        (temp["time"] < row.end_time)
        ].mean(numeric_only=True)

    df_row["Location"] = row.Location
    df_row["start_time"] = row.start_time
    df_row["end_time"] = row.end_time

    rows.append(df_row)

df_weather = pd.DataFrame(rows)

In [ ]:
df_final = gdf_cov.merge(df_weather, on = ['Location', 'start_time', 'end_time'], how = 'left')

In [ ]:
gdf_final = gpd.GeoDataFrame(df_final, geometry='geometry', crs="EPSG:4326")

### Add SEDAC score

In [ ]:
sedac_raw = gpd.read_parquet("/path/to/external_data/SEDAC/sedac_data.parquet")

In [ ]:
sedac_raw['geometry'] = sedac_raw.centroid

In [ ]:
sedac = (gpd
         .overlay(
             sedac_raw,
             gdf_final[['Location', 'geometry']],
             how='intersection')
         .groupby('Location', as_index=False)
         .agg(
             sedac_mean=('sedac', 'mean'),
             sedac_var=('sedac', 'var'))
        )

In [ ]:
gdf_final = gdf_final.merge(sedac, on = 'Location', how = 'left')

### Add population data

In [ ]:
def iso2_to_iso3(iso2_code):
    try: return pycountry.countries.get(alpha_2=iso2_code.upper()).alpha_3
    except: return None

gdf_final['iso3'] = gdf_final['iso2'].apply(iso2_to_iso3)

In [ ]:
results = []
for iso in tqdm(gdf_final['iso3'].unique()):
    
    country_gdf = gdf_final.loc[gdf_final.iso3 == iso, ['ID', 'geometry']].drop_duplicates()
    
    for sex in ['m', 'f']:
        for age in ['00', '05', '10', '15', '20', '25', '30', '35', '40', '45', '50', '55', '60', '65', '70', '75', '80', '85', '90']:
            for year in [2025]:
                
                file_name = f"{iso.lower()}_{sex}_{age}_{year}_CN_1km_R2025A_UA_v1.tif"
                storage_path = f"/path/to/external_data/WorldPop/Constrained_1km/"

                if not Path(f"{storage_path}{file_name}").exists():
                    iso_url = f'https://data.worldpop.org/GIS/AgeSex_structures/Global_2015_2030/R2025A/{year}/{iso}/v1/1km_ua/constrained/{file_name}'
                    response = requests.get(iso_url)
                    response.raise_for_status()
    
                    with open(f"{storage_path}{file_name}", 'wb') as f:
                        f.write(response.content)
                        print(f"Downloaded {sex}_{age} of {iso}!")
                
                stats = zonal_stats(country_gdf, f"{storage_path}{file_name}", stats="sum")
                country_gdf[f"{sex}_{age}_{year}"] = [s['sum'] for s in stats]
    
    results.append(country_gdf)
    
pop_raw = pd.concat(results)

In [ ]:
pop_raw["m_2025"] = pop_raw.loc[:, pop_raw.columns.str.startswith("m_")].sum(axis=1)
pop_raw["f_2025"] = pop_raw.loc[:, pop_raw.columns.str.startswith("f_")].sum(axis=1)
pop_raw["pop_2025"] = pop_raw["m_2025"] + pop_raw["f_2025"]

In [ ]:
gdf_final = gdf_final.merge(pop_raw.drop(columns = ['geometry']), on = 'ID', how = 'left')

### Get population density

In [ ]:
gdf_final['area_km'] = gdf_final.to_crs(3587).area / 1000000

In [ ]:
gdf_final["pop_dens_f_2025"] = gdf_final["f_2025"] / gdf_final["area_km"]
gdf_final["pop_dens_m_2025"] = gdf_final["m_2025"] / gdf_final["area_km"]
gdf_final["pop_dens_2025"] = gdf_final["pop_2025"] / gdf_final["area_km"]

### Get Timing variables

In [ ]:
gdf_final["hour_of_day_avg"] = (gdf_final["end_time"].dt.hour + gdf_final["start_time"].dt.hour)/2
gdf_final["day_of_week_start"] = gdf_final["start_time"].dt.day_name()

### Save data

In [ ]:
gdf_final.to_parquet('../midsave/control_vars.parquet')
gdf_final.to_csv('/path/to/external_data/control_vars.csv')

## Visualization

In [ ]:
africa_iso3 = [
    "DZA", "AGO", "BEN", "BWA", "BFA", "BDI", "CPV", "CMR", "CAF", "TCD",
    "COM", "COD", "COG", "CIV", "DJI", "EGY", "GNQ", "ERI", "SWZ", "ETH",
    "GAB", "GMB", "GHA", "GIN", "GNB", "KEN", "LSO", "LBR", "LBY", "MDG",
    "MWI", "MLI", "MRT", "MUS", "MYT", "MAR", "MOZ", "NAM", "NER", "NGA",
    "REU", "RWA", "SHN", "STP", "SEN", "SYC", "SLE", "SOM", "ZAF", "SSD",
    "SDN", "TZA", "TGO", "TUN", "UGA", "ESH", "ZMB", "ZWE"
]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

world_gdf[world_gdf.GID_0.isin(africa_iso3)].plot(ax=ax, color="lightgray", edgecolor="black")
gdf_final.plot(ax=ax, color="red", alpha=0.5)

plt.show()